# DemandIQ: Modeling with Direct Multi-Horizon Forecasting

Milestone 7 evaluates global direct multi-horizon models using the exact feature matrices and chronological folds produced by `04_feature_engineering.ipynb`. The target is transformed as `log1p(Sales)` for fitting and returned to the original Sales scale before metrics are calculated.

This milestone performs historical cross-validation only. It does not create final test predictions or a submission file. No random split, shuffle, future Sales, Customers, or Id is used.

## 1. Modeling Objective

The approved architecture is a global direct multi-horizon model: one model learns across stores and receives `ForecastHorizon` as a feature. Each validation fold predicts its complete 48-day horizon from a frozen forecast origin. Historical Sales features are already frozen by Milestone 6; this notebook consumes them unchanged.

Models are compared across all three chronological folds. No final model is selected from one fold alone, and no test forecast is generated here.

In [2]:
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'notebooks' / '04_feature_engineering.ipynb').is_file())
feature_notebook_path = project_root / 'notebooks' / '04_feature_engineering.ipynb'
with feature_notebook_path.open(encoding='utf-8') as notebook_file:
    feature_notebook = json.load(notebook_file)
feature_namespace = {'__name__': '__feature_engineering__', 'display': lambda *args, **kwargs: None}
for cell in feature_notebook['cells']:
    if cell.get('cell_type') == 'code':
        source = ''.join(cell.get('source', []))
        with redirect_stdout(StringIO()), redirect_stderr(StringIO()):
            exec(compile(source, str(feature_notebook_path), 'exec'), feature_namespace)

required_objects = ['X_train_by_fold', 'X_validation_by_fold', 'y_train_by_fold', 'y_validation_by_fold', 'X_test', 'fold_definitions', 'training_examples_by_fold', 'validation_examples_by_fold', 'model_feature_columns']
missing_objects = [name for name in required_objects if name not in feature_namespace]
assert not missing_objects, f'Missing Milestone 6 objects: {missing_objects}'
globals().update({name: feature_namespace[name] for name in required_objects})
fold_ids = list(fold_definitions['fold'])
print(f'Loaded exact Milestone 6 matrices from {feature_notebook_path.name}.')
print(f'Folds: {fold_ids}; shared feature count: {len(model_feature_columns)}; X_test: {X_test.shape}')

Loaded exact Milestone 6 matrices from 04_feature_engineering.ipynb.
Folds: [1, 2, 3]; shared feature count: 39; X_test: (41088, 39)


## 2. Load Feature Matrices

The objects below are the exact Milestone 6 outputs. `X_train_by_fold`, `X_validation_by_fold`, and `X_test` use one shared feature schema. Targets are kept separately and are never included in model inputs. The test matrix is inspected for schema compatibility only; no test prediction is made.

In [3]:
assert set(X_train_by_fold) == {1, 2, 3}
assert set(X_validation_by_fold) == {1, 2, 3}
assert set(y_train_by_fold) == {1, 2, 3}
assert set(y_validation_by_fold) == {1, 2, 3}
for fold in [1, 2, 3]:
    assert list(X_train_by_fold[fold].columns) == model_feature_columns
    assert list(X_validation_by_fold[fold].columns) == model_feature_columns
    assert len(X_train_by_fold[fold]) == len(y_train_by_fold[fold])
    assert len(X_validation_by_fold[fold]) == len(y_validation_by_fold[fold])
assert list(X_test.columns) == model_feature_columns
assert not {'Sales', 'Customers', 'Id', 'Date', 'HistoryOrigin'}.intersection(model_feature_columns)
print('Shared model schema verified:')
print(model_feature_columns)

Shared model schema verified:
['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'Quarter', 'IsWeekend', 'Promo2Active', 'CompetitionOpen', 'CompetitionAgeMonths', 'ForecastHorizon', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_median_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_median_14', 'rolling_std_14', 'rolling_mean_28', 'rolling_median_28', 'rolling_std_28', 'store_history_count']


## 3. Target Transformation

Models fit `y_log = log1p(Sales)`. This reduces the influence of very large stores and preserves zero Sales because `log1p(0) = 0`. Predictions are converted back with `expm1` and clipped at zero. Closed observations remain in training.

In [4]:
target_column = 'Sales'
def transform_target(sales):
    return np.log1p(np.asarray(sales, dtype='float64'))

def inverse_target(log_sales):
    return np.clip(np.expm1(np.asarray(log_sales, dtype='float64')), 0, None)

y_log_train_by_fold = {fold: transform_target(values) for fold, values in y_train_by_fold.items()}
y_log_validation_by_fold = {fold: transform_target(values) for fold, values in y_validation_by_fold.items()}
assert np.isclose(transform_target([0])[0], 0)
assert np.allclose(inverse_target(y_log_train_by_fold[1]), y_train_by_fold[1].to_numpy())
assert all((y_train_by_fold[fold] >= 0).all() for fold in [1, 2, 3])
print('Target transformation verified: log1p for fitting and non-negative expm1 inverse.')

Target transformation verified: log1p for fitting and non-negative expm1 inverse.


## 4. Evaluation Metrics

Primary metric is RMSPE on open stores only. Sales equal to zero are handled explicitly: they are counted and excluded from the percentage denominator because RMSPE is undefined at zero. MAE, RMSE, and R² are calculated on the original Sales scale.

In [5]:
def calculate_metrics(frame, predictions):
    actual = frame['Sales'].to_numpy(dtype='float64')
    predicted = np.asarray(predictions, dtype='float64')
    open_mask = frame['Open'].to_numpy() == 1
    zero_sales_count = int((open_mask & (actual == 0)).sum())
    rmspe_mask = open_mask & (actual != 0)
    if rmspe_mask.any():
        percentage_errors = (actual[rmspe_mask] - predicted[rmspe_mask]) / actual[rmspe_mask]
        rmspe = float(np.sqrt(np.mean(np.square(percentage_errors))))
    else:
        rmspe = np.nan
    return {
        'RMSPE_open': rmspe,
        'MAE': float(mean_absolute_error(actual, predicted)),
        'RMSE': float(np.sqrt(mean_squared_error(actual, predicted))),
        'R2': float(r2_score(actual, predicted)),
        'Open_rows': int(open_mask.sum()),
        'Open_zero_sales_excluded': zero_sales_count,
    }

def evaluation_frame(fold):
    return validation_examples_by_fold[fold].reset_index(drop=True)

assert calculate_metrics(pd.DataFrame({'Sales': [0, 10], 'Open': [1, 1]}), [2, 8])['Open_zero_sales_excluded'] == 1
print('Metric functions verified, including explicit open-store zero handling.')

Metric functions verified, including explicit open-store zero handling.


## 5. Naive Baseline

Model 0 uses the origin-based frozen seven-day rolling mean. It is repeated for every horizon and store, with a lag-1 fallback only if the rolling value is unavailable. It does not use validation-period Sales.

In [6]:
def naive_predictions(frame):
    baseline = frame['rolling_mean_7'].fillna(frame['lag_1']).fillna(0)
    return baseline.to_numpy(dtype='float64')

model_results = []
prediction_records = []
for fold in [1, 2, 3]:
    validation_frame = evaluation_frame(fold)
    predictions = naive_predictions(validation_frame)
    metrics = calculate_metrics(validation_frame, predictions)
    model_results.append({'Model': 'Naive origin rolling mean 7', 'Fold': fold, **metrics})
    prediction_records.append(pd.DataFrame({'Model': 'Naive origin rolling mean 7', 'Fold': fold, 'Sales': validation_frame['Sales'], 'Open': validation_frame['Open'], 'ForecastHorizon': validation_frame['ForecastHorizon'], 'Prediction': predictions}))
print('Naive baseline evaluated on all three folds.')

Naive baseline evaluated on all three folds.


## 6. Ridge Baseline

Ridge is a transparent regularized linear baseline. Each fold fits its imputation and one-hot preprocessing on `X_train` only, then applies the fitted transformer to `X_validation`. This avoids fitting preprocessing on validation data.

In [7]:
def make_preprocessor(frame):
    categorical = frame.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric = [column for column in frame.columns if column not in categorical]
    return ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical),
    ], remainder='drop')

ridge_models = {}
for fold in [1, 2, 3]:
    preprocessor = make_preprocessor(X_train_by_fold[fold])
    ridge_pipeline = Pipeline([('preprocessor', preprocessor), ('model', Ridge(alpha=10.0))])
    ridge_pipeline.fit(X_train_by_fold[fold], y_log_train_by_fold[fold])
    ridge_models[fold] = ridge_pipeline
    validation_frame = evaluation_frame(fold)
    predictions = inverse_target(ridge_pipeline.predict(X_validation_by_fold[fold]))
    metrics = calculate_metrics(validation_frame, predictions)
    model_results.append({'Model': 'Ridge log1p', 'Fold': fold, **metrics})
    prediction_records.append(pd.DataFrame({'Model': 'Ridge log1p', 'Fold': fold, 'Sales': validation_frame['Sales'], 'Open': validation_frame['Open'], 'ForecastHorizon': validation_frame['ForecastHorizon'], 'Prediction': predictions}))
print('Ridge evaluated on all three folds with fold-fitted preprocessing.')

C:\Users\pc\AppData\Local\Temp\ipykernel_9004\305650523.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = frame.select_dtypes(include=['object', 'category']).columns.tolist()
C:\Users\pc\AppData\Local\Temp\ipykernel_9004\305650523.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.or

Ridge evaluated on all three folds with fold-fitted preprocessing.


## 7. Tree-Based Model

Model 2 is a scikit-learn Random Forest regressor trained on the transformed target. It uses the same fold-fitted preprocessing and exact feature schema as Ridge. XGBoost is intentionally deferred.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

def make_tree_matrices(train_frame, validation_frame):
    train_matrix = pd.DataFrame(index=train_frame.index)
    validation_matrix = pd.DataFrame(index=validation_frame.index)
    categorical_columns = train_frame.select_dtypes(include=['object', 'category']).columns.tolist()
    for column in train_frame.columns:
        if column in categorical_columns:
            categories = pd.Index(train_frame[column].dropna().unique())
            category_codes = {value: code for code, value in enumerate(categories)}
            train_matrix[column] = train_frame[column].map(category_codes).fillna(-1).astype('float32')
            validation_matrix[column] = validation_frame[column].map(category_codes).fillna(-1).astype('float32')
        else:
            train_values = pd.to_numeric(train_frame[column], errors='coerce')
            validation_values = pd.to_numeric(validation_frame[column], errors='coerce')
            median = train_values.median()
            train_matrix[column] = train_values.fillna(median).astype('float32')
            validation_matrix[column] = validation_values.fillna(median).astype('float32')
    return train_matrix[model_feature_columns], validation_matrix[model_feature_columns]

tree_models = {}
for fold in [1, 2, 3]:
    tree_train, tree_validation = make_tree_matrices(X_train_by_fold[fold], X_validation_by_fold[fold])
    tree_model = HistGradientBoostingRegressor(
        max_iter=80,
        learning_rate=0.08,
        max_leaf_nodes=31,
        min_samples_leaf=100,
        l2_regularization=1.0,
        random_state=42,
    )
    tree_model.fit(tree_train, y_log_train_by_fold[fold])
    tree_models[fold] = tree_model
    validation_frame = evaluation_frame(fold)
    predictions = inverse_target(tree_model.predict(tree_validation))
    metrics = calculate_metrics(validation_frame, predictions)
    model_results.append({'Model': 'Histogram Gradient Boosting log1p', 'Fold': fold, **metrics})
    prediction_records.append(pd.DataFrame({'Model': 'Histogram Gradient Boosting log1p', 'Fold': fold, 'Sales': validation_frame['Sales'], 'Open': validation_frame['Open'], 'ForecastHorizon': validation_frame['ForecastHorizon'], 'Prediction': predictions}))
print('Histogram Gradient Boosting evaluated on all three folds with fold-fitted numeric/category preprocessing.')

C:\Users\pc\AppData\Local\Temp\ipykernel_9004\305650523.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = frame.select_dtypes(include=['object', 'category']).columns.tolist()


MemoryError: Unable to allocate 248. MiB for an array with shape (35, 930240) and data type float64

## 8. Chronological Fold Training

The result table below is assembled only after all folds and all models have been evaluated. There is no fitting on validation observations and no final test fit in this milestone.

In [ ]:
results_df = pd.DataFrame(model_results)
assert set(results_df['Fold']) == {1, 2, 3}
assert results_df.groupby('Model')['Fold'].nunique().eq(3).all()
assert len(prediction_records) == 9
predictions_df = pd.concat(prediction_records, ignore_index=True)
assert len(predictions_df) == 3 * sum(len(evaluation_frame(fold)) for fold in [1, 2, 3])
display(results_df.sort_values(['Model', 'Fold']))
print('All three chronological folds evaluated for every model.')

## 9. Model Comparison

Means and standard deviations summarize performance across folds. The primary ranking is lower `RMSPE_open`; MAE, RMSE, and R² provide secondary context. No single fold determines the candidate.

In [ ]:
comparison_table = results_df.groupby('Model').agg({
    'RMSPE_open': ['mean', 'std'],
    'MAE': ['mean', 'std'],
    'RMSE': ['mean', 'std'],
    'R2': ['mean', 'std'],
}).round(4)
comparison_table.columns = ['_'.join(column) for column in comparison_table.columns]
display(comparison_table.sort_values('RMSPE_open_mean'))
best_candidate = comparison_table['RMSPE_open_mean'].idxmin()
print(f'Best candidate by mean open-store RMSPE across all folds: {best_candidate}')
print('This is a comparison result only; no final test model or submission is produced.')

## 10. Horizon Error Analysis

Performance is examined by `ForecastHorizon` on the original Sales scale. This shows whether direct predictions degrade with forecast distance.

In [ ]:
horizon_rows = []
for keys, group in predictions_df.groupby(['Model', 'Fold', 'ForecastHorizon']):
    model, fold, horizon = keys
    metrics = calculate_metrics(group, group['Prediction'])
    horizon_rows.append({'Model': model, 'Fold': fold, 'ForecastHorizon': horizon, **metrics})
horizon_results = pd.DataFrame(horizon_rows)
horizon_summary = horizon_results.groupby(['Model', 'ForecastHorizon'])['RMSPE_open'].agg(['mean', 'std']).reset_index()
assert horizon_results['ForecastHorizon'].between(1, 48).all()
assert horizon_results.groupby('Model')['ForecastHorizon'].nunique().eq(48).all()
display(horizon_summary.head(15))
print('Horizon analysis covers all 48 horizons for every model.')

## 11. Closed/Open Analysis

Primary RMSPE is restricted to open stores, but closed stores remain in training and are examined separately. This section reports open-store metrics and closed-store prediction behavior without deleting closed observations.

In [ ]:
open_closed_rows = []
for (model, fold), group in predictions_df.groupby(['Model', 'Fold']):
    for store_state, state_mask in [('Open', group['Open'].eq(1)), ('Closed', group['Open'].eq(0))]:
        subset = group.loc[state_mask]
        if len(subset) == 0:
            continue
        open_closed_rows.append({
            'Model': model,
            'Fold': fold,
            'StoreState': store_state,
            'Rows': len(subset),
            'ActualSalesMean': subset['Sales'].mean(),
            'PredictionMean': subset['Prediction'].mean(),
            'PredictionMin': subset['Prediction'].min(),
            'PredictionMax': subset['Prediction'].max(),
            'MAE': mean_absolute_error(subset['Sales'], subset['Prediction']),
        })
open_closed_summary = pd.DataFrame(open_closed_rows)
assert (predictions_df['Prediction'] >= 0).all()
display(open_closed_summary)
print('Closed-store observations were retained; predictions are non-negative after inverse transformation and clipping.')

## 12. Leakage Checks and Modeling Conclusions

The modeling protocol uses only fold training matrices for fitting preprocessing and models. Validation Sales are targets for metrics only. Customers, Id, future Sales, and random splitting are excluded. The three-fold comparison is the evidence used for the next architecture decision; final test prediction is intentionally deferred.

In [ ]:
assert not {'Sales', 'Customers', 'Id'}.intersection(model_feature_columns)
assert all(list(X_train_by_fold[fold].columns) == list(X_test.columns) for fold in [1, 2, 3])
assert all(fold_definitions.loc[fold_definitions['fold'].eq(fold), 'validation_dates'].iloc[0][0] > fold_definitions.loc[fold_definitions['fold'].eq(fold), 'origin'].iloc[0] for fold in [1, 2, 3])
assert all((fold_definitions.loc[fold_definitions['fold'].eq(fold), 'origin'].iloc[0] + pd.Timedelta(days=48)) <= fold_definitions.loc[fold_definitions['fold'].eq(fold), 'validation_dates'].iloc[0][0] + pd.Timedelta(days=47) for fold in [1, 2, 3])
print('Modeling conclusions:')
print(f'- Models compared: {list(results_df[
].unique())}')
print(f'- Best mean RMSPE candidate: {best_candidate}')
print('- All three chronological folds were evaluated.')
print('- No final test predictions, submission file, or model artifact was created.')